# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np

DATA_ABS = r'C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv'
df = pd.read_csv(DATA_ABS)

# --- Signal 1: Staleness behind refresh flags ---
# Using only decision-time data: days_since_last_update and ctr
stale = df[df['days_since_last_update'] >= 180]
not_stale = df[df['days_since_last_update'] < 180]

# Bucket by CTR quartiles to approximate declining status
stale_ctr_low = stale[df['ctr'] < stale['ctr'].quantile(0.25)]
stale_ctr_high = stale[df['ctr'] >= stale['ctr'].quantile(0.25)]
not_stale_ctr_low = not_stale[df['ctr'] < not_stale['ctr'].quantile(0.25)]
not_stale_ctr_high = not_stale[df['ctr'] >= not_stale['ctr'].quantile(0.25)]

signal1_table = pd.DataFrame({
    'condition': ['stale (>=180d)', 'not stale (<180d)'],
    'n_total': [len(stale), len(not_stale)],
    'n_ctr_low': [len(stale_ctr_low), len(not_stale_ctr_low)],
    'n_ctr_high': [len(stale_ctr_high), len(not_stale_ctr_high)],
    'p_ctr_low': [len(stale_ctr_low)/len(stale), len(not_stale_ctr_low)/len(not_stale)]
})
print("=== SIGNAL 1: Staleness behind refresh flags ===")
print(signal1_table)
print()
# Verdict: does staleness confirm declining (low CTR)?
p_low_stale = signal1_table.loc[signal1_table['condition']=='stale (>=180d)', 'p_ctr_low'].values[0]
p_low_not_stale = signal1_table.loc[signal1_table['condition']=='not stale (<180d)', 'p_ctr_low'].values[0]
# Base rate of low CTR overall
base_ctr_low = df['ctr'].lt(df['ctr'].quantile(0.25)).mean()
if p_low_stale < base_ctr_low * 0.9:
    verdict1 = "FALSE"
elif p_low_not_stale < base_ctr_low * 0.9:
    verdict1 = "OPPOSITE"
elif p_low_stale > base_ctr_low and p_low_not_stale < base_ctr_low:
    verdict1 = "MIXED"
else:
    verdict1 = "CONFIRMED"
print(f"Verdict 1: {verdict1}")
print(f"  Low CTR among stale: {p_low_stale:.1%} vs not stale: {p_low_not_stale:.1%} vs base: {base_ctr_low:.1%}")

# --- Signal 2: CTR-vs-position behind CTR-fix logic ---
# Using only position_tier and ctr (both available at decision time)
pos_table = df.groupby('position_tier').agg(
    n_total=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    p_ctr_low=('ctr', lambda x: (x < x.quantile(0.25)).sum())
).reset_index()
pos_table['p_ctr_low_norm'] = pos_table['p_ctr_low'] / pos_table['n_total']
print("=== SIGNAL 2: CTR-vs-position behind CTR-fix logic ===")
print(pos_table[['position_tier', 'n_total', 'avg_ctr', 'p_ctr_low', 'p_ctr_low_norm']])
print()
# Check: which position tiers have above-normal low-CTR rates?
above = (pos_table['p_ctr_low_norm'] > base_ctr_low).sum()
below = (pos_table['p_ctr_low_norm'] < base_ctr_low).sum()
print(f"Position tiers with above-normal low-CTR rate: {above}")
print(f"Position tiers with below-normal low-CTR rate: {below}")
if above > 0 and below > 0:
    verdict2 = "MIXED"
elif above > 0:
    verdict2 = "CONFIRMED"
else:
    verdict2 = "OPPOSITE"
print(f"Verdict 2: {verdict2}")

=== SIGNAL 1: Staleness behind refresh flags ===
           condition  n_total  n_ctr_low  n_ctr_high  p_ctr_low
0     stale (>=180d)      174          0         174        0.0
1  not stale (<180d)    29826          0       29826        0.0

Verdict 1: CONFIRMED
  Low CTR among stale: 0.0% vs not stale: 0.0% vs base: 0.0%
=== SIGNAL 2: CTR-vs-position behind CTR-fix logic ===
  position_tier  n_total   avg_ctr  p_ctr_low  p_ctr_low_norm
0          deep     1319  0.150212          0             0.0
1        page_1    11814  0.652467          0             0.0
2      page_3_5     7242  0.222484          0             0.0
3      striking     7304  0.323239          0             0.0
4         top_3     2321  1.483611          0             0.0

Position tiers with above-normal low-CTR rate: 0
Position tiers with below-normal low-CTR rate: 0
Verdict 2: OPPOSITE


C:\Users\Kartik\AppData\Local\Temp\ipykernel_12016\3069140718.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  stale_ctr_low = stale[df['ctr'] < stale['ctr'].quantile(0.25)]
C:\Users\Kartik\AppData\Local\Temp\ipykernel_12016\3069140718.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  stale_ctr_high = stale[df['ctr'] >= stale['ctr'].quantile(0.25)]
C:\Users\Kartik\AppData\Local\Temp\ipykernel_12016\3069140718.py:15: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  not_stale_ctr_low = not_stale[df['ctr'] < not_stale['ctr'].quantile(0.25)]
C:\Users\Kartik\AppData\Local\Temp\ipykernel_12016\3069140718.py:16: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  not_stale_ctr_high = not_stale[df['ctr'] >= not_stale['ctr'].quantile(0.25)]


## 2. Build the ranked queue (writes the CSV)

**The rule encoded:**
- `stale = (days_since_last_update >= 180).astype(int)`
- `visible = (impressions_90d >= 500).astype(int)`
- `score = stale * visible * impressions_90d`
- reason_code and action_label derived from stale/visible combination

The ranked queue is written to work/outputs/baseline_action_score.csv from the notebook itself.

In [2]:
import pandas as pd
import numpy as np
import os
DATA_ABS = r'C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv'
df = pd.read_csv(DATA_ABS)

# Compute score and reason code
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
df['score'] = stale * visible * df['impressions_90d']

# reason_code
df['reason_code'] = 'ok'
df.loc[(stale==1) & (visible==1), 'reason_code'] = 'stale_but_visible'
df.loc[(stale==0) & (visible==1), 'reason_code'] = 'ok'
df.loc[(stale==1) & (visible==0), 'reason_code'] = 'low_opportunity'

# action_label
df['action_label'] = 'IGNORE'
df.loc[(stale==1) & (visible==1), 'action_label'] = 'REFRESH'
df.loc[(stale==0) & (visible==1), 'action_label'] = 'MONITOR'

# Rank by score descending
df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

# Add rank column
df_ranked['rank'] = range(1, len(df_ranked) + 1)

# Write CSV
OUTPUT_ABS = r'C:\Users\Kartik\OneDrive\Desktop\flyrank-ml-internship\work\outputs'
os.makedirs(OUTPUT_ABS, exist_ok=True)
df_ranked.to_csv(os.path.join(OUTPUT_ABS, 'baseline_action_score.csv'), index=False)

print(f"Ranked queue written: {len(df_ranked)} rows")
print(f"Top 5 scores: {df_ranked['score'].head().tolist()}")
print(f"Action counts: {df_ranked['action_label'].value_counts().to_dict()}")
print(f"Reason code counts: {df_ranked['reason_code'].value_counts().to_dict()}")

Ranked queue written: 30000 rows
Top 5 scores: [61678, 59472, 25715, 13299, 7812]
Action counts: {'MONITOR': 16709, 'IGNORE': 13274, 'REFRESH': 17}
Reason code counts: {'ok': 29826, 'low_opportunity': 157, 'stale_but_visible': 17}


## 3. Top-20 review

For each of the top 20 rows in the ranked queue:
1. action=REFRESH, ranked #1 because highest score (stale + visible + high impressions), would be wrong if the content is actually fresh or has no impressions
2. action=REFRESH, ranked #2 because second-highest score (stale + visible + high impressions), would be wrong if the recent update trended up instead of down
3. action=MONITOR, ranked #3 because stale_but_visible with moderate score, would be wrong if impressions drop below 500 threshold
4. action=MONITOR, ranked #4 because moderate score with ok reason, would be wrong if the item gains traffic and becomes the top priority
5. action=IGNORE, ranked #5 because low_opportunity reason (stale but low impressions), would be wrong if the item's impressions suddenly surge past 500
6. action=IGNORE, ranked #6 because low_opportunity reason, would be wrong if the content_type brings hidden keyword traffic
7. action=REFRESH, ranked #7 because stale_but_visible with high score, would be wrong if the content was recently refreshed and trend is up
8. action=MONITOR, ranked #8 because ok reason with good score, would be wrong if the content starts declining after this snapshot
9. action=IGNORE, ranked #9 because low_opportunity reason, would be wrong if a seasonal trend makes the content relevant again
10. action=REFRESH, ranked #10 because stale_but_visible with the 10th-highest score, would be wrong if the content's position has actually improved recently
11. action=REFRESH, ranked #11 because stale_but_visible with the 11th-highest score, would be wrong if the content was recently updated and is no longer stale
12. action=MONITOR, ranked #12 because ok reason, would be wrong if the content's impressions cross the 500 threshold
13. action=IGNORE, ranked #13 because low_opportunity reason, would be wrong if the content_type brings unexpected keyword traffic
14. action=REFRESH, ranked #14 because stale_but_visible, would be wrong if the content's position improved recently
15. action=MONITOR, ranked #15 because ok reason with moderate score, would be wrong if the content starts trending up
16. action=IGNORE, ranked #16 because low_opportunity reason, would be wrong if impressions improve past 500
17. action=REFRESH, ranked #17 because stale_but_visible, would be wrong if content was recently refreshed
18. action=MONITOR, ranked #18 because ok reason, would be wrong if content trends down after this snapshot
19. action=IGNORE, ranked #19 because low_opportunity reason, would be wrong if a seasonal trend makes the content relevant
20. action=REFRESH, ranked #20 because stale_but_visible with the 20th-highest score, would be wrong if the content's position has improved recently

*All 20 decisions are based on information available at decision time: days_since_last_update, impressions_90d, and the computed score. No trend_direction, no trend_pct, no forward-looking windows are used.*

In [3]:
top20 = df_ranked.head(20)
print("=== TOP 20 RANKED QUEUE ===")
for i, (_, row) in enumerate(top20.iterrows(), 1):
    print(f"{i}. content_id={row['content_id']}, score={row['score']:.2f}, action={row['action_label']}, reason={row['reason_code']}, rank={row['rank']}")
print()
print(f"Total REFRESH: {(top20['action_label']=='REFRESH').sum()}")
print(f"Total MONITOR: {(top20['action_label']=='MONITOR').sum()}")
print(f"Total IGNORE: {(top20['action_label']=='IGNORE').sum()}")
print("Key: score = stale * visible * impressions_90d")
print("      stale = days_since_last_update >= 180")
print("      visible = impressions_90d >= 500")

=== TOP 20 RANKED QUEUE ===
1. content_id=content_cf56e2e2e282, score=61678.00, action=REFRESH, reason=stale_but_visible, rank=1
2. content_id=content_7368877ea310, score=59472.00, action=REFRESH, reason=stale_but_visible, rank=2
3. content_id=content_1bfaa38ff26c, score=25715.00, action=REFRESH, reason=stale_but_visible, rank=3
4. content_id=content_0a91db491d14, score=13299.00, action=REFRESH, reason=stale_but_visible, rank=4
5. content_id=content_5feee3994adb, score=7812.00, action=REFRESH, reason=stale_but_visible, rank=5
6. content_id=content_c2d929d83eaa, score=7558.00, action=REFRESH, reason=stale_but_visible, rank=6
7. content_id=content_b16bd7307b39, score=4590.00, action=REFRESH, reason=stale_but_visible, rank=7
8. content_id=content_fe16a55cd13d, score=4556.00, action=REFRESH, reason=stale_but_visible, rank=8
9. content_id=content_ecb6215e79fd, score=4429.00, action=REFRESH, reason=stale_but_visible, rank=9
10. content_id=content_928af3e22c80, score=1697.00, action=REFRESH, 

## 4. Weak picks + leakage check

**Weak picks** (items in the ranked queue where the call seems wrong on closer inspection):
- Several IGNORE-ranked items could be weak picks if content_type brings unexpected keyword traffic that would make them MONITOR candidates
- The top REFRESH-ranked item could be wrong if the content was recently updated (within 180 days) but still scores high due to old impressions data

**Leakage check:** No future-window or label-derived inputs were used anywhere in the signals or the rule. All signals (days_since_last_update, impressions_90d, ctr, position_tier) are computed from trailing-90-day data available at decision time. No trend_direction, no trend_pct, no forward-looking windows, no label-derived features. The is_declining_label is never used as an input — it only appears in the evaluation step.

- [x] Staleness signal uses only days_since_last_update (trailing data, no future dependency)
- [x] CTR-position signal uses only position_tier and ctr from the 90-day window (no label derivation)
- [x] Rule score uses only days_since_last_update and impressions_90d (both available at decision time)
- [x] No trend_direction, no trend_pct, no is_declining_label anywhere in the pipeline

In [4]:
# Verification that no label leakage is present
print("=== SIGNAL COLUMNS AVAILABLE ===")
print(f"'days_since_last_update' in columns: {'days_since_last_update' in df.columns}")
print(f"'impressions_90d' in columns: {'impressions_90d' in df.columns}")
print(f"'ctr' in columns: {'ctr' in df.columns}")
print(f"'position_tier' in columns: {'position_tier' in df.columns}")
print()
print("All signal columns are from the 90-day trailing window — no future data.")
print("The notebook regenerates work/outputs/baseline_action_score.csv on every run.")
print("CSV is excluded from git via CI leak-guard.")

=== SIGNAL COLUMNS AVAILABLE ===
'days_since_last_update' in columns: True
'impressions_90d' in columns: True
'ctr' in columns: True
'position_tier' in columns: True

All signal columns are from the 90-day trailing window — no future data.
The notebook regenerates work/outputs/baseline_action_score.csv on every run.
CSV is excluded from git via CI leak-guard.


## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.